<img src ='https://i.imgur.com/HRhd2Y0.png'>


# Importa Dados

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import *

In [0]:
%sql
create database if not exists mkt_olist;

In [0]:
spark.read.format(
  'csv'
).options(
  header='true', inferschema='true'
).load(
  '/FileStore/tables/olist_orders_dataset.csv'
).write.mode(
  'overwrite'
).saveAsTable(
  'mkt_olist.mkt_olist_orders'
)

In [0]:
spark.read.format(
  'csv'
).options(
  header='true', inferschema='true'
).load(
  '/FileStore/tables/olist_order_payments_dataset.csv'
).write.mode(
  'overwrite'
).saveAsTable(
  'mkt_olist.mkt_olist_pagamentos'
)

# Manipulação dos dados

In [0]:
# Read Orders Table
df_orders = spark.table(
  'mkt_olist.mkt_olist_orders'
)

# Read Payment Table
df_payment = spark.table(
  'mkt_olist.mkt_olist_pagamentos'
)



In [0]:
df_payment.groupBy(
  'payment_type' 
).agg(
  avg('payment_value').alias('avg_payment')
).display()

In [0]:
%sql
select payment_type
,avg(payment_value) as avg_payment
from mkt_olist.mkt_olist_pagamentos
group by all
--group by 1
--group by payment_type

payment_type,avg_payment
boleto,145.03443540234633
not_defined,0.0
credit_card,163.31902063935996
voucher,65.70335411255414
debit_card,142.57017004578165


In [0]:
spark.sql(
  """
    select payment_type
    ,avg(payment_value) as avg_payment
    from mkt_olist.mkt_olist_pagamentos
    group by all
  """
).display()

payment_type,avg_payment
boleto,145.03443540234633
not_defined,0.0
credit_card,163.31902063935996
voucher,65.70335411255414
debit_card,142.57017004578165


In [0]:
df_orders.join(
  df_payment, ['order_id'], 'left'
).filter(
  col('order_status')=='invoiced'
).withColumn(
  'Mes', lpad(month(to_date(col('order_approved_at'))),2,'0')
).display()

In [0]:
%sql
select a.*
,b.*
,lpad(month(to_date(a.order_approved_at)),2,'0') as Mes
from mkt_olist.mkt_olist_orders as a
left join mkt_olist.mkt_olist_pagamentos as b
on a.order_id = b.order_id
where a.order_status = "invoiced"

order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_id,payment_sequential,payment_type,payment_installments,payment_value,Mes
76361b050296df7e327ae497ed570427,573b8654cb5e34f524e337bc94ce387c,invoiced,2018-05-21T15:39:08.000+0000,2018-05-22T16:57:54.000+0000,null,null,2018-06-07T00:00:00.000+0000,76361b050296df7e327ae497ed570427,1,credit_card,8,504.64,05
442be05839341a530c60503334ac0488,d12dca16ecd5d626b790f6d85f7c20b2,invoiced,2017-09-21T12:50:17.000+0000,2017-09-21T13:04:41.000+0000,null,null,2017-10-17T00:00:00.000+0000,442be05839341a530c60503334ac0488,1,credit_card,1,97.12,09
29eb6d873f1087b4afd226c0abc162a4,18395a5d9973f59eb27c9f8c5efbb9e7,invoiced,2017-09-10T17:39:52.000+0000,2017-09-10T17:50:35.000+0000,null,null,2017-09-28T00:00:00.000+0000,29eb6d873f1087b4afd226c0abc162a4,1,credit_card,2,66.59,09
bc6bbd933ea8d286847d6eb88c6083ed,20209b475c58be96dcfc7d9f773198d3,invoiced,2018-08-03T16:31:05.000+0000,2018-08-04T16:30:14.000+0000,null,null,2018-08-22T00:00:00.000+0000,bc6bbd933ea8d286847d6eb88c6083ed,1,credit_card,1,33.38,08
9ee84e2bfcbb0e4aa06217c6ecbab36e,159ad0f6cbd047fb360f05bb09e6cfe4,invoiced,2017-11-09T13:33:20.000+0000,2017-11-10T03:10:46.000+0000,null,null,2017-11-29T00:00:00.000+0000,9ee84e2bfcbb0e4aa06217c6ecbab36e,1,boleto,1,106.16,11
308389e2d91267e2713b8e3ae5a52f0e,b19e123e70a2fd33a077217a1b4db0d5,invoiced,2018-03-15T18:37:12.000+0000,2018-03-17T03:09:00.000+0000,null,null,2018-04-03T00:00:00.000+0000,308389e2d91267e2713b8e3ae5a52f0e,1,boleto,1,886.96,03
c9fff6f42450201f1c5b500726598c70,07d39c9c43a03208a3329192bfa03a01,invoiced,2017-09-21T13:12:56.000+0000,2017-09-23T02:25:07.000+0000,null,null,2017-10-20T00:00:00.000+0000,c9fff6f42450201f1c5b500726598c70,1,boleto,1,311.19,09
9e99a131a8798c3985381f4f7c96d56e,d9287300a4a22b52cebc3186370089bb,invoiced,2018-01-19T12:52:52.000+0000,2018-01-19T13:14:42.000+0000,null,null,2018-02-16T00:00:00.000+0000,9e99a131a8798c3985381f4f7c96d56e,1,boleto,1,164.01,01
fe67d9fbc13fc8b281cfab9b09f60607,2fcf059f0e80fc034f5bc5e878e9115b,invoiced,2017-12-15T17:46:39.000+0000,2017-12-15T18:31:43.000+0000,null,null,2018-01-16T00:00:00.000+0000,fe67d9fbc13fc8b281cfab9b09f60607,1,credit_card,1,214.57,12
d1ca0c1c5f78ecd17ab721d805629eb7,9739ff8d93cf3032dc57d7c0f20d0de5,invoiced,2017-06-02T09:45:29.000+0000,2017-06-06T13:25:42.000+0000,null,null,2017-06-23T00:00:00.000+0000,d1ca0c1c5f78ecd17ab721d805629eb7,1,boleto,1,310.98,06


In [0]:
spark.sql(
  """
  select a.*
  ,b.*
  ,lpad(month(to_date(a.order_approved_at)),2,'0') as Mes
  from mkt_olist.mkt_olist_orders as a
  left join mkt_olist.mkt_olist_pagamentos as b
  on a.order_id = b.order_id
  where a.order_status = "invoiced"
  """
).display()

order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_id,payment_sequential,payment_type,payment_installments,payment_value,Mes
76361b050296df7e327ae497ed570427,573b8654cb5e34f524e337bc94ce387c,invoiced,2018-05-21T15:39:08.000+0000,2018-05-22T16:57:54.000+0000,null,null,2018-06-07T00:00:00.000+0000,76361b050296df7e327ae497ed570427,1,credit_card,8,504.64,05
442be05839341a530c60503334ac0488,d12dca16ecd5d626b790f6d85f7c20b2,invoiced,2017-09-21T12:50:17.000+0000,2017-09-21T13:04:41.000+0000,null,null,2017-10-17T00:00:00.000+0000,442be05839341a530c60503334ac0488,1,credit_card,1,97.12,09
29eb6d873f1087b4afd226c0abc162a4,18395a5d9973f59eb27c9f8c5efbb9e7,invoiced,2017-09-10T17:39:52.000+0000,2017-09-10T17:50:35.000+0000,null,null,2017-09-28T00:00:00.000+0000,29eb6d873f1087b4afd226c0abc162a4,1,credit_card,2,66.59,09
bc6bbd933ea8d286847d6eb88c6083ed,20209b475c58be96dcfc7d9f773198d3,invoiced,2018-08-03T16:31:05.000+0000,2018-08-04T16:30:14.000+0000,null,null,2018-08-22T00:00:00.000+0000,bc6bbd933ea8d286847d6eb88c6083ed,1,credit_card,1,33.38,08
9ee84e2bfcbb0e4aa06217c6ecbab36e,159ad0f6cbd047fb360f05bb09e6cfe4,invoiced,2017-11-09T13:33:20.000+0000,2017-11-10T03:10:46.000+0000,null,null,2017-11-29T00:00:00.000+0000,9ee84e2bfcbb0e4aa06217c6ecbab36e,1,boleto,1,106.16,11
308389e2d91267e2713b8e3ae5a52f0e,b19e123e70a2fd33a077217a1b4db0d5,invoiced,2018-03-15T18:37:12.000+0000,2018-03-17T03:09:00.000+0000,null,null,2018-04-03T00:00:00.000+0000,308389e2d91267e2713b8e3ae5a52f0e,1,boleto,1,886.96,03
c9fff6f42450201f1c5b500726598c70,07d39c9c43a03208a3329192bfa03a01,invoiced,2017-09-21T13:12:56.000+0000,2017-09-23T02:25:07.000+0000,null,null,2017-10-20T00:00:00.000+0000,c9fff6f42450201f1c5b500726598c70,1,boleto,1,311.19,09
9e99a131a8798c3985381f4f7c96d56e,d9287300a4a22b52cebc3186370089bb,invoiced,2018-01-19T12:52:52.000+0000,2018-01-19T13:14:42.000+0000,null,null,2018-02-16T00:00:00.000+0000,9e99a131a8798c3985381f4f7c96d56e,1,boleto,1,164.01,01
fe67d9fbc13fc8b281cfab9b09f60607,2fcf059f0e80fc034f5bc5e878e9115b,invoiced,2017-12-15T17:46:39.000+0000,2017-12-15T18:31:43.000+0000,null,null,2018-01-16T00:00:00.000+0000,fe67d9fbc13fc8b281cfab9b09f60607,1,credit_card,1,214.57,12
d1ca0c1c5f78ecd17ab721d805629eb7,9739ff8d93cf3032dc57d7c0f20d0de5,invoiced,2017-06-02T09:45:29.000+0000,2017-06-06T13:25:42.000+0000,null,null,2017-06-23T00:00:00.000+0000,d1ca0c1c5f78ecd17ab721d805629eb7,1,boleto,1,310.98,06


In [0]:
df_orders_vs_payment = df_orders.join(
  df_payment, ['order_id'], 'left'
).filter(
  col('order_status')=='invoiced'
).withColumn(
  'Mes', lpad(month(to_date(col('order_approved_at'))),2,'0')
).groupBy(
  'Mes'
).agg(
  sum('payment_value').cast('decimal(32,2)').alias('Faturamento')
).orderBy(
  col('Mes').desc()
)

df_orders_vs_payment.display()

In [0]:
%sql
select mes
,cast(sum(payment_value) as decimal(32,2)) as Faturamento
from(
select a.*
  ,b.*
  ,lpad(month(to_date(a.order_approved_at)),2,'0') as Mes
  from mkt_olist.mkt_olist_orders as a
  left join mkt_olist.mkt_olist_pagamentos as b
  on a.order_id = b.order_id
  where a.order_status = "invoiced"
)
group by all
order by mes desc

mes,Faturamento
12,2117.07
11,5090.97
10,6062.77
09,4256.66
08,7380.86
07,5508.29
06,1520.57
05,9700.94
04,4030.89
03,9972.62


In [0]:
spark.sql(
  """
  select mes
  ,cast(sum(payment_value) as decimal(32,2)) as Faturamento
  from(
  select a.*
    ,b.*
    ,lpad(month(to_date(a.order_approved_at)),2,'0') as Mes
    from mkt_olist.mkt_olist_orders as a
    left join mkt_olist.mkt_olist_pagamentos as b
    on a.order_id = b.order_id
    where a.order_status = "invoiced"
  )
  group by all
  order by mes desc
  """
).display()

mes,Faturamento
12,2117.07
11,5090.97
10,6062.77
09,4256.66
08,7380.86
07,5508.29
06,1520.57
05,9700.94
04,4030.89
03,9972.62
